In [1]:
"""
Strike clustering via Ward hierarchical clustering, weighted by |qty|.

Design choices (per prior discussion):
  - Invariant preserved: total signed quantity (sum of clust_qty == sum of init_qty)
  - Clustering: Ward linkage, with |qty| as observation weights
    (implemented via observation duplication, since scipy.linkage does not
    accept a weights argument)
  - Cluster representative: the strike with the largest |qty| in the cluster
    (snap-to-real-strike; ties broken by proximity to the weighted centroid)
  - Strikes not chosen as representatives get clust_qty = 0
"""

from __future__ import annotations

import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import linkage, fcluster


def strike_clustering(position: pd.Series, n: int) -> pd.DataFrame:
    """
    Reduce a portfolio of option positions to at most `n` strikes by Ward
    hierarchical clustering, weighted by the absolute quantity at each strike.

    Parameters
    ----------
    position : pd.Series
        Index = strike (float), value = signed quantity (int).
        Positive = long, negative = short. Zero quantities are kept in
        init_qty but cannot pull the clustering.
    n : int
        Maximum number of distinct strikes in the output (must be >= 1).

    Returns
    -------
    pd.DataFrame
        Indexed by the original strikes, sorted ascending.
        Columns:
          - init_qty  : original signed quantity at that strike
          - clust_qty : post-clustering signed quantity. Exactly one strike
                        per cluster (the representative) carries the cluster's
                        summed quantity; the others are 0.

        Invariant: clust_qty.sum() == init_qty.sum()
    """
    # ---- 1. validate & normalize input ------------------------------------
    if not isinstance(position, pd.Series):
        raise TypeError("`position` must be a pandas Series.")
    if n < 1:
        raise ValueError("`n` must be >= 1.")

    # sort by strike for deterministic, readable output
    pos = position.sort_index().astype(int)
    strikes = pos.index.to_numpy(dtype=float)
    qty = pos.to_numpy(dtype=int)
    abs_qty = np.abs(qty)
    num_strikes = len(strikes)

    out = pd.DataFrame(
        {"init_qty": qty, "clust_qty": 0},
        index=pd.Index(strikes, name=pos.index.name or "strike"),
    )

    # ---- 2. trivial cases -------------------------------------------------
    # nothing to cluster
    if num_strikes == 0:
        return out

    # already <= n distinct strikes: pass-through
    if num_strikes <= n:
        out["clust_qty"] = qty
        return out

    # all-zero book: no risk anywhere, just keep the first n strikes as zeros
    # (clust_qty stays 0 everywhere — the invariant 0 == 0 holds trivially)
    if abs_qty.sum() == 0:
        return out

    # ---- 3. build the weighted observation set ---------------------------
    # Strikes with qty == 0 carry no risk and must not influence the merge
    # decisions, but they still need a cluster label so we can absorb them
    # somewhere. Strategy: cluster only the nonzero strikes, then assign each
    # zero-qty strike to its nearest representative afterwards.
    nonzero_mask = abs_qty > 0
    nz_strikes = strikes[nonzero_mask]
    nz_qty = qty[nonzero_mask]
    nz_abs = abs_qty[nonzero_mask]
    num_nz = len(nz_strikes)

    # If the number of nonzero strikes is already <= n we don't need to merge
    # them — each becomes its own cluster, and zero-qty strikes get absorbed
    # into the nearest. We still go through the assignment path below.
    if num_nz <= n:
        labels_nz = np.arange(num_nz)  # one cluster per nonzero strike
    else:
        # Observation duplication: repeat strike[i] |qty[i]| times so that
        # Ward's variance objective is weighted by |qty|. Equivalent to
        # weighted Ward in the math, exact for integer weights.
        expanded = np.repeat(nz_strikes, nz_abs).reshape(-1, 1)

        # Ward linkage on the expanded set.
        Z = linkage(expanded, method="ward")

        # Cut the dendrogram to obtain exactly n clusters.
        expanded_labels = fcluster(Z, t=n, criterion="maxclust")

        # Collapse expanded labels back to one label per original strike.
        # Every duplicate of a given strike receives the same cluster label
        # (they sit at the same coordinate), so taking the first occurrence
        # is safe.
        first_idx = np.concatenate(([0], np.cumsum(nz_abs)[:-1]))
        labels_nz = expanded_labels[first_idx]

    # ---- 4. pick a representative strike per cluster ---------------------
    # Heaviest |qty| in the cluster wins; ties broken by proximity to the
    # weighted centroid of the cluster.
    unique_labels = np.unique(labels_nz)
    representatives: dict[int, float] = {}

    for lab in unique_labels:
        member_mask = labels_nz == lab
        member_strikes = nz_strikes[member_mask]
        member_abs = nz_abs[member_mask]

        # weighted centroid of the cluster
        centroid = np.average(member_strikes, weights=member_abs)

        # primary key: -|qty| (so argmin gives heaviest)
        # secondary key: distance to centroid (so argmin gives closest on tie)
        order = np.lexsort((np.abs(member_strikes - centroid), -member_abs))
        rep = member_strikes[order[0]]
        representatives[int(lab)] = float(rep)

    # ---- 5. assign zero-qty strikes to nearest representative ------------
    # Build the full label array aligned with `strikes`.
    full_labels = np.empty(num_strikes, dtype=int)
    full_labels[nonzero_mask] = labels_nz

    if (~nonzero_mask).any():
        rep_array = np.array(list(representatives.values()))
        rep_labels = np.array(list(representatives.keys()))
        for i in np.where(~nonzero_mask)[0]:
            nearest = np.argmin(np.abs(rep_array - strikes[i]))
            full_labels[i] = rep_labels[nearest]

    # ---- 6. aggregate signed qty into clust_qty at each representative ---
    clust_qty = np.zeros(num_strikes, dtype=int)
    for lab, rep_strike in representatives.items():
        member_mask = full_labels == lab
        total = int(qty[member_mask].sum())
        rep_pos = int(np.where(strikes == rep_strike)[0][0])
        clust_qty[rep_pos] = total

    out["clust_qty"] = clust_qty

    # ---- 7. sanity check on the invariant --------------------------------
    assert out["clust_qty"].sum() == out["init_qty"].sum(), (
        "Total quantity not preserved: "
        f"{out['clust_qty'].sum()} vs {out['init_qty'].sum()}"
    )

    return out


# ----------------------------------------------------------------------------
# Demo / smoke test
# ----------------------------------------------------------------------------
if __name__ == "__main__":
    # Example from the dialogue: 8 strikes, mixed signs, a heavy 105 and an
    # isolated 200.
    position = pd.Series(
        data=[2, -1, 3, 50, -2, 1, 2, -1],
        index=[90.0, 95.0, 100.0, 105.0, 130.0, 135.0, 140.0, 200.0],
        name="qty",
    )
    position.index.name = "strike"

    print("Input positions:")
    print(position.to_string())
    print()

    for n in [1, 2, 3, 5, 8, 10]:
        result = strike_clustering(position, n=n)
        print(f"=== n = {n} ===")
        print(result.to_string())
        print(
            f"init_qty sum = {result['init_qty'].sum()}, "
            f"clust_qty sum = {result['clust_qty'].sum()}"
        )
        print()

Input positions:
strike
90.0      2
95.0     -1
100.0     3
105.0    50
130.0    -2
135.0     1
140.0     2
200.0    -1

=== n = 1 ===
        init_qty  clust_qty
strike                     
90.0           2          0
95.0          -1          0
100.0          3          0
105.0         50         54
130.0         -2          0
135.0          1          0
140.0          2          0
200.0         -1          0
init_qty sum = 54, clust_qty sum = 54

=== n = 2 ===
        init_qty  clust_qty
strike                     
90.0           2          0
95.0          -1          0
100.0          3          0
105.0         50         54
130.0         -2          0
135.0          1          0
140.0          2          0
200.0         -1          0
init_qty sum = 54, clust_qty sum = 54

=== n = 3 ===
        init_qty  clust_qty
strike                     
90.0           2          0
95.0          -1          0
100.0          3          0
105.0         50         54
130.0         -2          1
135